In [1]:
from astroplan import Observer
from astropy.coordinates import EarthLocation
import astropy.units as u
from astropy.time import Time

from icalendar import Calendar, Event
import pytz
from datetime import datetime,timedelta

import pandas as pd

from tqdm.notebook import tqdm  # 推荐在 notebook 里用这个


In [2]:
def ny_to_utc(ny_datetime):
    """
    将纽约本地时间转换为 UTC 的 Astropy Time 对象
    
    参数:
    ny_datetime: datetime 对象 (无时区或纽约时区)
    
    返回:
    astropy.time.Time 对象 (UTC)
    """
    # 确保有时区信息
    if ny_datetime.tzinfo is None:
        ny_tz = pytz.timezone('America/New_York')
        ny_datetime = ny_tz.localize(ny_datetime)
    
    # 转换为 UTC
    utc_datetime = ny_datetime.astimezone(pytz.utc)
    
    return Time(utc_datetime, format='datetime', scale='utc')

In [6]:
schedule_dict = {
    "Site": [
        "RM",
        "KP",
        "CT",
        "KP",
        "CT",
    ],
    "Date": [
        "2026-03-02",
        "2026-03-03",
        "2026-03-13",
        "2026-03-29",
        "2026-03-29",
    ],
    "Observer": [
        "Garima",
        "Yong", 
        "Antonio",
        "Ava",
        "Alba",
    ],
}
    

In [7]:
df = pd.DataFrame(schedule_dict)
df

,Site,Date,Observer
0,RM,2026-03-02,Garima
1,KP,2026-03-03,Yong
2,CT,2026-03-13,Antonio
3,KP,2026-03-29,Ava
4,CT,2026-03-29,Alba


In [9]:
cal = Calendar()
cal.add('prodid', '-//Observation Calendar//')
cal.add('version', '2.0')
cal.add('x-wr-timezone', 'UTC')  # 明确声明时区

sites = df["Site"]
dates = df["Date"]
observers = df["Observer"]

for site, date, observer in tqdm(zip(sites, dates, observers), total = len(sites)):

    # trandslate the site acronym to full name
    if site == "CT":
        site_name = "Cerro Tololo"
    elif site == "KP":
        site_name = "Kitt Peak"
    elif site == "RM":
        site_name = "Roque de los Muchachos, La Palma"
    else:
        raise ValueError(f"Site {site} is not supported yet!")


    observer_site = Observer.at_site(site_name)
    
    observe_date = Time(f"{date} 12:00:00", format="iso")  # define the observation date at noon

    make_list_date = ny_to_utc(Time(f"{date} 09:00:00", format="iso").datetime) # always make the list at 9AM before the obs date
    make_list_date = make_list_date.datetime.replace(tzinfo=pytz.utc) - timedelta(days=1)

    sunset_time = observer_site.sun_set_time(observe_date, which='next').datetime.replace(tzinfo=pytz.utc)
    
    sunrise_time = observer_site.sun_rise_time(observe_date, which='next').datetime.replace(tzinfo=pytz.utc)

    # sometimes the sunset will be later than the sunrise:
    # sunset: datetime.datetime(2025, 6, 10, 2, 27, 19, 986049, tzinfo=<UTC>)
    #sunrise: datetime.datetime(2025, 6, 9, 12, 24, 21, 148462, tzinfo=<UTC>)
    # which is not correct! 
    # this only happens for KP
    if sunset_time > sunrise_time:
        sunrise_time = observer_site.sun_rise_time(observe_date + 1*u.day, which='next').datetime.replace(tzinfo=pytz.utc)

    # first add the calender for making the observation list
    make_list_event = Event()
    make_list_event.add("summary", f"Make {site} Observation List")
    make_list_event.add("dtstart", make_list_date)
    make_list_event.add("dtend", make_list_date + timedelta(hours=1))
    make_list_event.add('dtstamp', datetime.utcnow().replace(tzinfo=pytz.utc))
    make_list_event.add('uid', f"{site}_{make_list_date}")
    make_list_event.add('valarm', {'action': 'DISPLAY',
                                   'trigger': timedelta(minutes=-15),
                                   'description': ""})
    cal.add_component(make_list_event)

    # next add the calender for the calibration observation
    calibration_event = Event()
    calibration_event.add("summary", f"Take {site} Calibration Frames")
    calibration_event.add("dtstart", sunset_time - timedelta(hours=1.5))
    calibration_event.add("dtend", sunset_time)
    calibration_event.add('dtstamp', datetime.utcnow().replace(tzinfo=pytz.utc))
    calibration_event.add('uid', f"{site}_{sunset_time - timedelta(hours=1.5)}")
    calibration_event.add('valarm', {'action': 'DISPLAY', 
                                     'trigger': timedelta(minutes=-15),
                                     'description': ""})
    cal.add_component(calibration_event)
    
    # last add the calender for the observation
    observation_event = Event()
    observation_event.add("summary", f"{observer} {site} Observation")
    observation_event.add("dtstart", sunset_time)
    observation_event.add("dtend", sunrise_time)
    observation_event.add('dtstamp', datetime.utcnow().replace(tzinfo=pytz.utc))
    observation_event.add('uid', f"{site}_{sunset_time}")
    observation_event.add('valarm', {'action': 'DISPLAY',
                                     'trigger': timedelta(minutes=0),
                                     'description': ""})
    cal.add_component(observation_event)


with open('Observation_calendar.ics', 'wb') as f:
    f.write(cal.to_ical())
    #f.write(event.to_ical())

  0%|          | 0/5 [00:00<?, ?it/s]